In [0]:
# MAGIC %md
# MAGIC # Notebook 00_ingesta - Capa Bronce (Lakehouse)
# MAGIC **Grupo:** G5 - EnergiaVisible  
# MAGIC **Sector:** Energía Eléctrica  
# MAGIC **Fuente oficial:** SiMEM - XM (Operador del Mercado de Energía Mayorista)  
# MAGIC **Dataset:** Demanda Real Nacional  
# MAGIC **URL de descarga:** https://www.simem.co/  
# MAGIC **Fecha de descarga:** 2026-10-13  
# MAGIC **Objetivo:** Ingestar datos crudos desde Unity Catalog Volumes y persistirlos como tabla Delta en Capa Bronce sin transformaciones analíticas.

In [0]:
import os

# 1. Listar el contenido de la carpeta para obtener el nombre exacto del archivo
carpeta = "/Volumes/workspace/default/vol_datos_g5"
archivos = dbutils.fs.ls(carpeta)

# Tomamos automáticamente el primer archivo que termine en .csv
archivo_csv = [f.path for f in archivos if f.name.endswith(".csv")][0]
print(f"Archivo detectado: {archivo_csv}")

# 2. Cargar con Spark usando la ruta detectada
df_crudo = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(archivo_csv)

# 3. Mostrar primeros 5 registros
display(df_crudo.limit(5))

Archivo detectado: dbfs:/Volumes/workspace/default/vol_datos_g5/Demandarealnacionale48c1a76-524a-4464-b495-ac17071141ec.csv


CodigoVariable,FechaHora,CodigoSICAgente,TipoMercado,Version,Valor,UnidadMedida,CodigoDuracion
DdaReal,2026-09-26T13:00:00.000Z,QIEC,No Regulado,TXF,2132.24,kWh,PT1H
DdaReal,2026-09-26T11:00:00.000Z,PEEC,No Regulado,TXF,324.05,kWh,PT1H
DdaReal,2026-09-26T08:00:00.000Z,DRUC,No Regulado,TXF,12179.69,kWh,PT1H
DdaReal,2026-09-26T10:00:00.000Z,RTQC,No Regulado,TXF,34587.69,kWh,PT1H
DdaReal,2026-09-26T05:00:00.000Z,TRPC,No Regulado,TXF,14.99,kWh,PT1H


In [0]:
total_filas = df_crudo.count()
total_columnas = len(df_crudo.columns)

print(f"Total de registros (filas): {total_filas}")
print(f"Total de columnas: {total_columnas}")

if total_filas >= 50000:
    print("✅ CUMPLE con el requisito de Big Data (>= 50.000 registros)")
else:
    print(f"⚠️ Cantidad de filas: {total_filas}. (Si pesa > 20 MB también cumple según el protocolo)")

Total de registros (filas): 63168
Total de columnas: 8
✅ CUMPLE con el requisito de Big Data (>= 50.000 registros)


In [0]:
# Muestra los nombres de columnas y tipos de datos inferidos
df_crudo.printSchema()

root
 |-- CodigoVariable: string (nullable = true)
 |-- FechaHora: timestamp (nullable = true)
 |-- CodigoSICAgente: string (nullable = true)
 |-- TipoMercado: string (nullable = true)
 |-- Version: string (nullable = true)
 |-- Valor: double (nullable = true)
 |-- UnidadMedida: string (nullable = true)
 |-- CodigoDuracion: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Calcula cuántos nulos tiene cada columna del dataset
df_nulos = df_crudo.select(*(spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_crudo.columns))
display(df_nulos)

CodigoVariable,FechaHora,CodigoSICAgente,TipoMercado,Version,Valor,UnidadMedida,CodigoDuracion
0,0,0,0,0,0,0,0


In [0]:
# Guardar en Delta Lake (Capa Bronce)
nombre_tabla = "default.bronce_energia_g5"

df_crudo.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(nombre_tabla)

print(f"🚀 ¡Tabla Delta '{nombre_tabla}' creada y persistida con éxito en Capa Bronce!")

🚀 ¡Tabla Delta 'default.bronce_energia_g5' creada y persistida con éxito en Capa Bronce!
